# modal

> Jobs in Modal sandboxes: a CPU, or one to eight T4, L4, A10, L40S, A100, H100, H200 or B200 GPUs, for up to a day; the job's files on a Modal Volume, an image Modal builds once from sysone's requirements, the log followed, the job stopped from here, and sessions resumed from the volume.

In [ ]:
#| default_exp modal

In [ ]:
#| export
import email, json, os, time, zipfile
from pathlib import Path

from sysone.cloud import (Modal, CPU, GPU, Job, LocalPath, ModalVolume, machine, stage, submit, poll, cancel, collect, unpack_outputs, save_job)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging, sys, tempfile
logging.getLogger("transformers").setLevel(logging.ERROR)
from sysone.cloud import remote, jobs, describe, hardware, as_data
from sysone.inference import Decider

[Modal](https://modal.com) runs containers on demand and bills them by the second. A sysone job there is a Modal *sandbox*, a container that runs one command until it ends, for up to 24 hours, which suits the [runner](40_cloud.ipynb#the-jobs-files): it is started once, left alone, and looked at again from here, now or from another session tomorrow.

1. The job's folder (entry, data, wheel, runner) is uploaded to a Modal **Volume**, `sysone-jobs` by default, under the job's id. The sandbox mounts the volume at `/jobs`, so the runner works in `/jobs/<job id>` and everything it writes there stays: the log, the outputs, the checkpoints, the feature cache.
2. The sandbox's **image** is Debian with Python 3.12 and sysone's requirements, read from the job's wheel; Modal builds it the first time and keeps it, so later jobs with the same requirements start at once. The runner installs the wheel itself, which is quick with every dependency in place.
3. The sandbox runs the runner with its output in `job.log`. A poll asks Modal whether the sandbox has ended and reads the log: from the running sandbox, and from the volume once it has stopped. At the end it brings `outputs.tar.gz` back from the volume.

What Modal allows shapes less than on Colab or Kaggle: secrets reach a sandbox (the Hub token, so a job pushes its export and its checkpoints itself), `job.stop()` stops it, and the volume outlives every sandbox, so a resumed job reads its predecessor's checkpoint where it was written. Two things to know:

- **Cost.** Modal has no free GPU quota: the Starter plan gives $30 of credit a month, and a GPU needs a payment method on file. At the time of writing, a T4 costs about $0.59 an hour, an L4 $0.80, an A100 (40 GB) $2.10, an H100 $3.95. A sandbox's CPU and memory are billed at three times their list price, so the default 4 cores and 16 GB add about $0.95 an hour, beside the GPU; `Modal(cpu=..., memory_gb=...)` changes them.
- **Preemption.** A sandbox with a GPU can be preempted. Its job then ends as `error`, but its last [checkpoint](05b_checkpoint.ipynb) is on the volume, and `job.resume()` picks up from it (see [sessions](#sessions)).

The client is an extra, `pip install "sysone[modal]"`, and signs in once with `modal setup` (or `login("modal")`, see [auth](43_auth.ipynb#modal)).

In [ ]:
#| export
class ModalError(RuntimeError): "Modal refused, or failed at, an operation"

MODAL_ROOT = "/jobs"              # where a sandbox mounts the jobs' volume: a job works in /jobs/<job id>
MODAL_GRACE = 600                 # seconds a sandbox lives past a job's max_hours: the runner stops the entry, then packs its outputs
MODAL_MAX = 24 * 3600             # Modal's longest sandbox

def _modal():
    "The Modal client, imported when a job needs it"
    try: import modal
    except ImportError: raise ModalError('the Modal client is not installed: pip install "sysone[modal]", then sign in with `modal setup`') from None
    return modal

## Hardware and data

Modal names its GPUs by model, `"A100:2"` for two on one machine, up to eight (four A10s). The A10G of other clouds is Modal's A10, and an `A100` may come with 80 GB at the price of 40. A sandbox gets `cpu` cores and `memory_gb` beside its GPU, whichever it has.

Local data travels with the job to the volume, whatever its size. Data already on a Modal Volume is named `modal:volume/path` and read where it is: on the jobs' volume through its mount, on another volume through a mount of its own under `/volumes`.

In [ ]:
#| export
MODAL_GPUS = {"T4": "T4", "L4": "L4", "A10": "A10", "A10G": "A10", "L40S": "L40S", "A100": "A100", "H100": "H100", "H200": "H200",
              "B200": "B200", "RTX6000": "RTX-PRO-6000"}

@machine.register
def machine_modal_cpu(b:Modal, hw:CPU) -> str: return "cpu"

@machine.register
def machine_modal_gpu(b:Modal, hw:GPU) -> str:
    if hw.name not in MODAL_GPUS: raise ValueError(f"Modal's GPUs are {', '.join(MODAL_GPUS)}, not {hw.name}")
    most = 4 if MODAL_GPUS[hw.name] == "A10" else 8
    if hw.count > most: raise ValueError(f"a Modal machine has at most {most} {hw.name}")
    return MODAL_GPUS[hw.name] + (f":{hw.count}" if hw.count > 1 else "")

@stage.register
def stage_modal_local(d:LocalPath, b:Modal, var:str) -> dict: return {"ship": {var: d.path}}     # uploaded with the job

@stage.register
def stage_modal_volume(d:ModalVolume, b:Modal, var:str) -> dict:
    name, _, path = d.ref.partition("/")
    mount = MODAL_ROOT if name == b.volume else f"/volumes/{name}"
    return {"env": {var: f"{mount}/{path}".rstrip("/")}} | ({} if name == b.volume else {"modal_volumes": {name: mount}})

In [ ]:
mb = Modal()
test_eq([machine(mb, hardware(h)) for h in ("cpu", "t4", "a10g", "l40s", "a100x2", "h100x8")], ["cpu", "T4", "A10", "L40S", "A100:2", "H100:8"])
test_fail(lambda: machine(mb, GPU("A10", 8)), contains="at most 4")
test_fail(lambda: machine(mb, hardware("v5e-1")), contains="has no")
test_eq(stage(as_data("modal:sysone-jobs/datasets/emotions.jsonl"), mb, "SYSONE_DATA"), {"env": {"SYSONE_DATA": "/jobs/datasets/emotions.jsonl"}})
test_eq(stage(as_data("modal:corpora/news"), mb, "SYSONE_DATA"), {"env": {"SYSONE_DATA": "/volumes/corpora/news"}, "modal_volumes": {"corpora": "/volumes/corpora"}})

## The image

`job_requirements` reads what sysone needs from the job's wheel (its `Requires-Dist`, without the optional extras), or names the release when the job installs one, and adds the job's own packages. `modal_image` puts them in Debian slim with the backend's Python, or in `Modal(image=...)`, a registry image to start from. The definition is the same for every job with the same requirements, which is what lets Modal keep one build for all of them.

In [ ]:
#| export
def job_requirements(folder, spec:dict) -> list:
    "What a job's image installs: sysone's requirements (read from the job's wheel, or its release) and the job's other packages"
    pkgs = list(spec.get("install") or [])
    if not pkgs: return []
    first, rest = pkgs[0], pkgs[1:]
    if not first.endswith(".whl"): return [first] + rest
    with zipfile.ZipFile(Path(folder) / first) as z:
        meta = next(n for n in z.namelist() if n.endswith(".dist-info/METADATA"))
        reqs = email.message_from_bytes(z.read(meta)).get_all("Requires-Dist") or []
    return [r for r in reqs if "extra ==" not in r] + rest

def modal_image(b:Modal, reqs:list):
    "Debian slim with the backend's Python (or the registry image `b.image`), with `reqs` installed"
    m = _modal()
    img = m.Image.from_registry(b.image, add_python=b.python) if b.image else m.Image.debian_slim(python_version=b.python)
    return img.uv_pip_install(*reqs) if reqs else img

In [ ]:
script = Path(tempfile.mkdtemp()) / "train.py"
script.write_text("""import os
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
learn.export(job_output())
open(os.path.join(os.path.dirname(job_output()), "token_seen"), "w").write(str(os.environ.get("HF_TOKEN") == "hf_not_a_real_token"))
""")
os.environ |= {"SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}
wj = remote(script, on="modal", hw="t4", data="fixtures/typed_decisions_tiny.jsonl", track=True, dry_run=True)
reqs = job_requirements(wj.folder, json.loads((Path(wj.folder) / "job.json").read_text()))
assert any(r.startswith("torch") for r in reqs) and any(r.startswith("transformers") for r in reqs) and "mlflow>=3.16" in reqs
test_eq([r for r in reqs if "extra" in r], [])
reqs[:4]

['torch>=2.8', 'transformers>=5.0', 'peft>=0.20', 'accelerate>=1.0']

## Starting and following a job

`submit` uploads the folder, then creates the sandbox: the runner as its command (its output appended to `job.log`), the image, the GPU, `cpu` and `memory_gb`, the jobs' volume at `/jobs` and any other volume the data names, the secrets, and a lifetime of `max_hours` plus `MODAL_GRACE`, which leaves the runner time to pack the outputs after it stops the entry (a job without `max_hours` gets Modal's longest, a day). The sandbox's id is the job's `ref`.

`poll` asks Modal whether the sandbox has ended and reads the log: from the running sandbox, whose files are always current, and once it has stopped from the volume, which Modal commits as the sandbox stops. An ended job's status is the runner's `status.json` on the volume, read again for a few seconds should the commit lag; a sandbox that stopped without writing one (preempted, out of memory, stopped from here) ends as `error`, or `stopped`. `cancel` terminates the sandbox, and `collect` unpacks the outputs the last poll brought back.

In [ ]:
#| export
def _secrets(b:Modal) -> dict:
    "The values of the backend's secrets on this machine (the Hub login for HF_TOKEN), for the names that have one"
    out = {}
    for name in b.secrets:
        v = os.environ.get(name)
        if v is None and name == "HF_TOKEN":
            from huggingface_hub import get_token
            v = get_token()
        if v: out[name] = v
    return out

def _read(vol, path:str) -> bytes|None:
    "A file on a volume, or None while it is not there"
    try: return b"".join(vol.read_file(path))
    except Exception: return None

@submit.register
def submit_modal(b:Modal, job:Job) -> Job:
    m, folder, d = _modal(), Path(job.folder), f"{MODAL_ROOT}/{job.id}"
    spec = json.loads((folder / "job.json").read_text())
    vol = m.Volume.from_name(b.volume, create_if_missing=True, version=2)
    with vol.batch_upload() as up: up.put_directory(str(folder), f"/{job.id}")
    vols = {MODAL_ROOT: vol} | {mount: m.Volume.from_name(name) for name, mount in (spec.get("modal_volumes") or {}).items()}
    sec = _secrets(b)
    life = min(spec["max_seconds"] + MODAL_GRACE, MODAL_MAX) if spec.get("max_seconds") else MODAL_MAX
    sb = m.Sandbox.create("sh", "-c", f"python {d}/sysone_job.py >> {d}/job.log 2>&1", app=m.App.lookup(b.app, create_if_missing=True),
                          image=modal_image(b, job_requirements(folder, spec)), gpu=None if job.machine == "cpu" else job.machine,
                          cpu=b.cpu, memory=int(b.memory_gb * 1024), timeout=int(life), volumes=vols,
                          secrets=[m.Secret.from_dict(sec)] if sec else [], workdir=d, tags={"sysone-job": job.id})
    job.ref, job.status = sb.object_id, "running"
    return job

@poll.register
def poll_modal(b:Modal, job:Job) -> str:
    m, folder = _modal(), Path(job.folder)
    vol, sb = m.Volume.from_name(b.volume), m.Sandbox.from_id(job.ref)
    rc = sb.poll()
    data = None
    if rc is None:
        try: data = sb.filesystem.read_bytes(f"{MODAL_ROOT}/{job.id}/job.log")
        except Exception: pass
    if data is None: data = _read(vol, f"{job.id}/job.log") or b""
    text = data[job.offset:].decode("utf-8", errors="replace")
    job.offset = max(job.offset, len(data))
    with open(folder / "job.log", "a") as f: f.write(text)
    if rc is not None:                   # ended: its status and outputs, from the volume
        for wait in (0, 5, 10):
            time.sleep(wait)
            st = _read(vol, f"{job.id}/outputs/status.json")
            if st is not None: break
        if st is not None:
            with open(folder / "outputs.tar.gz", "wb") as f: vol.read_file_into_fileobj(f"{job.id}/outputs.tar.gz", f)
            s = json.loads(st)
            job.status, job.url = s["status"], s.get("url") or job.url
        elif job.status != "stopped": job.status = "error"
    return text

@cancel.register
def cancel_modal(b:Modal, job:Job) -> Job:
    if job.ref: _modal().Sandbox.from_id(job.ref).terminate()
    job.status = "stopped"
    return job

@collect.register
def collect_modal(b:Modal, job:Job, dest:Path) -> Path:
    tb = Path(job.folder) / "outputs.tar.gz"
    if not tb.exists(): raise ModalError(f"{job.id} is {job.status}: its outputs come back when it ends (job.wait())")
    return unpack_outputs(tb, dest)

## A job on a stand-in Modal

The round trip runs here against a stand-in for the Modal client: a `modal` module whose volumes are folders, whose images are lists of their build steps, and whose sandboxes are processes of this machine running the command with the volumes' mount points mapped to those folders. It writes down every call, with the names of the secrets but never their values:

In [ ]:
FAKE = r'''import contextlib, json, os, shlex, shutil, signal, subprocess, sys
ROOT = os.environ["FAKE_MODAL_ROOT"]
def _note(*a):
    with open(os.path.join(ROOT, "calls.log"), "a") as f: f.write(json.dumps(a, default=str) + "\n")
class App:
    @staticmethod
    def lookup(name, create_if_missing=False, **kw): _note("app", name); return App()
class Secret:
    def __init__(self, env): self.env = env
    @staticmethod
    def from_dict(env_dict={}): _note("secret", sorted(env_dict)); return Secret(dict(env_dict))
class Image:
    def __init__(self, steps): self.steps = steps
    @staticmethod
    def debian_slim(python_version=None, force_build=False): return Image([["debian_slim", python_version]])
    @staticmethod
    def from_registry(tag, add_python=None, **kw): return Image([["from_registry", tag, add_python]])
    def uv_pip_install(self, *packages, **kw): return Image(self.steps + [["uv_pip_install", list(packages)]])
class _Upload:
    def __init__(self, root): self.root = root
    def put_directory(self, local, remote): shutil.copytree(local, os.path.join(self.root, remote.lstrip("/")), dirs_exist_ok=True)
class Volume:
    def __init__(self, name): self.name, self.root = name, os.path.join(ROOT, "volumes", name)
    @staticmethod
    def from_name(name, create_if_missing=False, version=None, **kw):
        v = Volume(name)
        if not os.path.isdir(v.root):
            if not create_if_missing: raise FileNotFoundError(f"no volume {name}")
            os.makedirs(v.root); _note("volume", name, version)
        return v
    @contextlib.contextmanager
    def batch_upload(self, force=False): yield _Upload(self.root)
    def _file(self, path):
        p = os.path.join(self.root, path.lstrip("/"))
        if not os.path.isfile(p): raise FileNotFoundError(path)
        return p
    def read_file(self, path):
        with open(self._file(path), "rb") as f: yield f.read()
    def read_file_into_fileobj(self, path, fileobj):
        with open(self._file(path), "rb") as f: shutil.copyfileobj(f, fileobj)
class _Files:
    def __init__(self, sb): self.sb = sb
    def read_bytes(self, path):
        with open(self.sb.local(path), "rb") as f: return f.read()
class Sandbox:
    def __init__(self, sid): self.object_id, self.dir, self.filesystem = sid, os.path.join(ROOT, "sandboxes", sid), _Files(self)
    def local(self, path):
        for mount, root in json.load(open(os.path.join(self.dir, "mounts.json"))).items(): path = path.replace(mount, root)
        return path
    @staticmethod
    def create(*args, app=None, image=None, gpu=None, timeout=300, volumes={}, secrets=(), workdir=None, cpu=None, memory=None, tags=None, **kw):
        os.makedirs(os.path.join(ROOT, "sandboxes"), exist_ok=True)
        sb = Sandbox(f"sb-{len(os.listdir(os.path.join(ROOT, 'sandboxes'))) + 1}"); os.makedirs(sb.dir)
        mounts = {mount: v.root for mount, v in volumes.items()}
        json.dump(mounts, open(os.path.join(sb.dir, "mounts.json"), "w"))
        _note("sandbox", sb.object_id, list(args), {"gpu": gpu, "timeout": timeout, "cpu": cpu, "memory": memory, "workdir": workdir, "tags": tags,
                                                    "image": image.steps if image else None, "mounts": sorted(mounts)})
        job = sb.local(os.path.join(workdir, "job.json"))          # the job names its inputs by their paths in the sandbox: map them too
        if os.path.exists(job):
            s = open(job).read()
            for mount, root in mounts.items(): s = s.replace(f'"{mount}/', f'"{root}/')
            open(job, "w").write(s)
        bin_ = os.path.join(ROOT, "bin"); os.makedirs(bin_, exist_ok=True)
        py = os.path.join(bin_, "python")              # this machine's Python, in its environment (a symlink would lose the venv)
        if not os.path.exists(py): open(py, "w").write(f"#!/bin/sh\nexec {shlex.quote(sys.executable)} \"$@\"\n"); os.chmod(py, 0o755)
        env = dict(os.environ, PATH=bin_ + os.pathsep + os.environ["PATH"])
        for s in secrets or (): env.update(s.env)
        rc = os.path.join(sb.dir, "rc")
        p = subprocess.Popen(["sh", "-c", shlex.join([sb.local(a) for a in args]) + f"; echo $? > {shlex.quote(rc)}"], env=env,
                             cwd=sb.local(workdir) if workdir else None, start_new_session=True)
        open(os.path.join(sb.dir, "pid"), "w").write(str(p.pid))
        return sb
    @staticmethod
    def from_id(sandbox_id, client=None): return Sandbox(sandbox_id)
    def poll(self):
        f = os.path.join(self.dir, "rc")
        s = open(f).read().strip() if os.path.exists(f) else ""
        return int(s) if s else None
    def terminate(self, wait=False):
        if self.poll() is None:
            try: os.killpg(int(open(os.path.join(self.dir, "pid")).read()), signal.SIGKILL)
            except ProcessLookupError: pass
            open(os.path.join(self.dir, "rc"), "w").write("137")
        _note("terminate", self.object_id)
'''
fake_pkg, fake_root = Path(tempfile.mkdtemp()), Path(tempfile.mkdtemp())
(fake_pkg / "modal").mkdir(); (fake_pkg / "modal" / "__init__.py").write_text(FAKE)
sys.path.insert(0, str(fake_pkg)); sys.modules.pop("modal", None)
os.environ |= {"FAKE_MODAL_ROOT": str(fake_root), "HF_TOKEN": "hf_not_a_real_token"}

In [ ]:
enc = str(Path("fixtures/tiny-modernbert").resolve())
mj = remote(script, on="modal", hw="t4", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": enc}, max_hours=2, install=False)
test_eq((mj.machine, mj.ref), ("T4", "sb-1"))
mj.wait(every=1, logs=False)
test_eq(mj.status, "complete")
calls = [json.loads(l) for l in (fake_root / "calls.log").read_text().splitlines()]
sbx = next(c for c in calls if c[0] == "sandbox")
test_eq({k: sbx[3][k] for k in ("gpu", "timeout", "cpu", "memory", "workdir", "mounts")},
        {"gpu": "T4", "timeout": 2 * 3600 + MODAL_GRACE, "cpu": 4.0, "memory": 16384, "workdir": f"/jobs/{mj.id}", "mounts": ["/jobs"]})
test_eq(next(c for c in calls if c[0] == "secret")[1], ["HF_TOKEN"])                         # the name, never the value
assert "hf_not_a_real_token" not in (fake_root / "calls.log").read_text()
exp = mj.fetch()
test_eq((Path(mj.outputs) / "token_seen").read_text(), "True")                             # the token reached the job
assert [r for r in mj.progress() if r["event"] == "log"]
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(set(Decider.load(exp, device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))

`job.stop()` ends a sandbox at once (Modal kills it), and the job is `stopped`:

In [ ]:
sleepy = Path(tempfile.mkdtemp()) / "sleepy.py"; sleepy.write_text("import time; time.sleep(120)\n")
sj = remote(sleepy, on="modal", hw="cpu", install=False)
sj.stop()
test_eq((sj.status, json.loads((fake_root / "calls.log").read_text().splitlines()[-1])), ("stopped", ["terminate", sj.ref]))
jobs(backend="modal")

job,backend,hardware,entry,status,elapsed,step,loss,last
modal-20261029-162656-1945,modal,T4,train.py,complete,0.3 min,2/2,1.0059,0 s left · MPS 0.00 GB
modal-20261029-162711-a6f3,modal,cpu,sleepy.py,stopped,0.0 min,,,


## Sessions

A sandbox lives at most a day, and a training that needs longer runs over several jobs (see [sessions](40_cloud.ipynb#sessions)): with `max_hours`, training stops `wrap_up_minutes` before the limit with a checkpoint, and the job ends `paused`. `job.resume()` starts a new sandbox that reads the old job's checkpoint where it was written, on the volume (`modal:sysone-jobs/<old job>/outputs/checkpoint`), with the feature cache beside it, which the runner keeps on the volume too. Nothing passes through this machine, and it works the same after a preemption, since the checkpoint on the volume does not depend on the sandbox having ended well. A job can also push its checkpoints to the Hub (`checkpoints="hf:..."`), which a job on another backend can resume from.

On the stand-in, a head trained on cached vectors, made to pause two seconds into its fit, then resumed in a second sandbox:

In [ ]:
slow = Path(tempfile.mkdtemp()) / "slow_train.py"
slow.write_text("""import os, time, torch
from transformers import TrainerCallback
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
class Slow(TrainerCallback):
    def on_step_end(self, *a, **k): time.sleep(float(os.environ.get("STEP_SECONDS", "0.25")))
torch.manual_seed(0)
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", cache="masks", logging_steps=1, disable_tqdm=True, callbacks=[Slow()])
if os.environ.get("PAUSE_AFTER"): learn.stop_at(time.time() + float(os.environ["PAUSE_AFTER"]), wrap_up=0)
learn.fit_one_cycle(3, lr=1e-3)
learn.export(job_output())
""")
m1 = remote(slow, on="modal", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": enc, "PAUSE_AFTER": "2"}, install=False)
m1.wait(every=1, logs=False)
test_eq(m1.status, "paused")
m2 = m1.resume(env={"ENCODER": enc})
test_eq(json.loads((Path(m2.folder) / "job.json").read_text())["env"]["SYSONE_RESUME"], f"/jobs/{m1.id}/outputs/checkpoint")
m2.wait(every=1, logs=False)
test_eq((m2.status, m2.parent), ("complete", m1.id))
r = next(r for r in m2.progress() if r["event"] == "resume")
test_eq(r["cache"], True)
print(describe(r))
test_eq(set(Decider.load(m2.fetch(), device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))

resuming fit 1 at step 2/24, with the feature cache


In [ ]:
sys.path.remove(str(fake_pkg)); sys.modules.pop("modal", None)
del os.environ["FAKE_MODAL_ROOT"], os.environ["HF_TOKEN"]          # the real cells below use your Modal and Hub logins

## On real Modal

These cells need the Modal client and a sign-in (`modal setup`); they are flagged `modal`, so the tests skip them unless run with `--flags modal`. The tiny training runs on a CPU sandbox, which the free credit covers; the first job also builds the image, which takes a few minutes once. Then a training that pauses in one sandbox and resumes in a second, which reads the first one's checkpoint on the volume:

In [ ]:
#| modal
real = remote(script, on="modal", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"}, max_hours=1)
real.wait(every=20)
test_eq(real.status, "complete")
Decider.load(real.fetch(), device="cpu").predict(rec["state"], rec["questions"])

In [ ]:
#| modal
rm1 = remote(slow, on="modal", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base", "PAUSE_AFTER": "60", "STEP_SECONDS": "2"})
rm1.wait(every=20)
test_eq(rm1.status, "paused")
rm2 = rm1.resume(env={"ENCODER": "answerdotai/ModernBERT-base", "STEP_SECONDS": "2"})
rm2.wait(every=20)
test_eq(rm2.status, "complete")
[describe(r) for r in rm2.progress() if r["event"] in ("resume", "checkpoint")][:3]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()